In [5]:
import pandas as pd
import numpy as np
from collections import defaultdict

df = pd.read_csv("ecommerce_customer_churn_dataset.csv")
df = df.dropna() #just gonna get rid of any datapoints that are missing data for simplicity...

def engagement(row):
    score = row["Login_Frequency"] + row["Session_Duration_Avg"] + row["Pages_Per_Session"]
    if score < np.percentile(df["Login_Frequency"] + df["Session_Duration_Avg"] + df["Pages_Per_Session"], 33):
        return "Low"
    elif score < np.percentile(df["Login_Frequency"] + df["Session_Duration_Avg"] + df["Pages_Per_Session"], 66):
        return "Medium"
    else:
        return "High"

def risk(row):
    if row["Cart_Abandonment_Rate"] > np.percentile(df["Cart_Abandonment_Rate"], 70):
        return "AtRisk"
    else:
        return "Active"

df["State1"] = df.apply(engagement, axis=1)
df["State2"] = df.apply(risk, axis=1)

transitions = defaultdict(lambda: defaultdict(int))

for i in range(len(df)):
    s1 = df["State1"].iloc[i]
    s2 = df["State2"].iloc[i]
    transitions[s1][s2] += 1

transition_matrix = {}

for s in ["Low", "Medium", "High"]:
    total = sum(transitions[s].values())
    transition_matrix[s] = {}
    for s2 in ["Active", "AtRisk"]:
        transition_matrix[s][s2] = transitions[s][s2] / total if total > 0 else 0

for k, v in transition_matrix.items():
    print(k, v)

Low {'Active': 0.3786882481874895, 'AtRisk': 0.6213117518125105}
Medium {'Active': 0.7521122000675904, 'AtRisk': 0.24788779993240959}
High {'Active': 0.9633192044343006, 'AtRisk': 0.03668079556569938}
